In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import os

from utils import set_seed

import mintflow

# Get dataset

In [4]:
set_seed(0)
batch_size = 4096

In [5]:
adata = sc.read(
    f"/data2/a330d/datasets/cosmx/crc_wt_cosmx/crc_202.h5ad",
    backup_url=f"https://zenodo.org/records/15574384/files/242.h5ad?download=1"
)
# adata.obsm = {} # NOTE: only some strange PCA embeddings are stored in obsm, which we don't need
adata.obs_names_make_unique()

label_to_coarse = {
    "epi1": "Epithelial",
    "epi2": "Epithelial",
    "epi3": "Epithelial",
    "epi4": "Epithelial",
    
    "fib1": "Fibroblast",
    "fib2": "Fibroblast",
    
    "EC": "Endothelial",
    "SMC": "Smooth_muscle",
    
    "BC": "B_cell",
    "PC_IgA": "Plasma_cell",
    "PC_IgG": "Plasma_cell",
    "PC_IgM": "Plasma_cell",
    
    "TC": "T_cell",
    
    "mye1": "Myeloid",
    "mye2": "Myeloid",
    
    "mast": "Mast_cell",
}

adata.obs["coarse_type"] = adata.obs['ist'].map(label_to_coarse)
labels_key = 'coarse_type'
domains_key = 'typ'
batch_key = 'sid'
adata = adata[~adata.obs[domains_key].isna()] # NOTE: Interesting to annotate?
adata = adata[~adata.obs[labels_key].isna()]

sc.pp.filter_cells(adata, min_counts=3)
sc.pp.filter_genes(adata, min_counts=3)

/data/a330d/miniforge3/envs/mintflow/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:174: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["n_counts"] = number


In [6]:
adata.obs[labels_key] = adata.obs[labels_key].astype('category')
adata.obsm['spatial'] = adata.obs[['CenterX_global_px', 'CenterY_global_px']].values
adata.layers['counts'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, layer='counts', flavor='seurat_v3', n_top_genes=2000, subset=True)

In [7]:
from cellina._spatial_utils import spatial_neighbors

n_neighbors = 50
spatial_neighbors(adata, bandwidth=np.inf, max_neighbours=n_neighbors, standardize=False)

In [8]:
adata.obs['neighbor'] = adata.obsp['spatial_connectivities'][:,0].toarray().astype(np.float32)

In [9]:
from scipy.sparse import csr_matrix

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obsm['spatial_x'] = adata.obsp['spatial_connectivities'] @ adata.X / n_neighbors
# float32
adata.obsm['spatial_x'] = csr_matrix(adata.obsm['spatial_x']).astype(np.float32)

## Data splits

In [10]:
split = "ood"

# Get holdout indices
if split == "random":
    fraction = 0.1
    n_cells = adata.n_obs
    n_holdout = int(n_cells * fraction)

    # Randomly choose cells
    test_idx = np.random.choice(n_cells, n_holdout, replace=False)

elif split == "ood":
    # OOD: Non-ref, non-epithelial
    #is_tumor_region  = adata.obs["typ"].str.contains("CRC|TVA", regex=True)
    is_tumor_region  = adata.obs["typ"].str.contains("CRC", regex=True)
    is_non_epi = adata.obs["coarse_type"] != "Epithelial"

    # Combine for test set
    test_mask = (is_tumor_region) & (is_non_epi)
    test_idx = np.where(test_mask)[0]
else:
    raise ValueError(f"Unknown split: {split}")

# Get train/val indices
all_idx = np.arange(adata.n_obs)
trainval_idx = np.setdiff1d(all_idx, test_idx)

In [11]:
# Set 'is_holdout' to False by default, then True for selected cells
adata.obs['is_holdout'] = False
adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

In [12]:
from sklearn.model_selection import train_test_split

validation_size = 0.1
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=validation_size,
    random_state=0,
    shuffle=True,
)

In [49]:
adata.X = adata.layers['counts'].copy() # NOTE: use raw counts for training

In [50]:
adata.write_h5ad(f"/data2/a330d/datasets/cosmx/crc_wt_cosmx/crc_202_mintflow.h5ad")

# Mintflow

In [ ]:
path_anndata = '/data2/a330d/datasets/cosmx/crc_wt_cosmx/crc_202_mintflow.h5ad'
adata = sc.read_h5ad(path_anndata)

In [ ]:
# -------------------------
# Define training mask
# -------------------------

train_mask = (
    (adata.obs["typ"].str.contains("REF|TVA", regex=True)) |
    (
        (adata.obs["typ"].str.contains("CRC", regex=True)) &
        (adata.obs["coarse_type"] == "Epithelial")
    )
)

adata_train = adata[train_mask].copy()
adata_eval = adata[~train_mask].copy()

print("Training cells:", adata_train.n_obs)
print("Held-out CRC non-epithelial:", adata_eval.n_obs)

Training cells: 384099
Held-out CRC non-epithelial: 36595


In [ ]:
import anndata as ad

max_cells = 50000
subsets = []

for niche in adata_train.obs["typ"].unique():
    ad_n = adata_train[adata_train.obs["typ"] == niche].copy()
    
    if ad_n.n_obs > max_cells:
        idx = np.random.choice(ad_n.obs_names, max_cells, replace=False)
        ad_n = ad_n[idx].copy()
    
    subsets.append(ad_n)

adata_train = ad.concat(subsets, axis=0)

In [ ]:
adata_train.X = adata_train.X.tocsr()
adata_eval.X  = adata_eval.X.tocsr()

In [5]:
outdir = "/data2/a330d/datasets/mintflow_single_section"
os.makedirs(outdir, exist_ok=True)

adata_train.obs["sliceID"] = "slide1"
adata_train.obs["batchID"] = "slide1"

adata_train.obs["sliceID"] = adata_train.obs["sliceID"].astype("category")
adata_train.obs["batchID"] = adata_train.obs["batchID"].astype("category")

train_file = os.path.join(outdir, "slide1_train.h5ad")
adata_train.write_h5ad(train_file)


In [ ]:
adata_eval.obs["sliceID"] = "slide1_eval"
adata_eval.obs["batchID"] = "slide1_eval"

adata_eval.obs["sliceID"] = adata_eval.obs["sliceID"].astype("category")
adata_eval.obs["batchID"] = adata_eval.obs["batchID"].astype("category")

eval_file = os.path.join(outdir, "crc_other_eval.h5ad")
adata_eval.write_h5ad(eval_file)

## Training

In [82]:
config_data_train, config_data_evaluation, config_model, config_training = \
    mintflow.get_default_configurations(
        num_tissue_sections_training=1,
        num_tissue_sections_evaluation=1
    )

In [83]:
outdir = "/data2/a330d/datasets/mintflow_single_section"
os.makedirs(outdir, exist_ok=True)
train_file = os.path.join(outdir, "slide1_train.h5ad")
eval_file = os.path.join(outdir, "crc_other_eval.h5ad")

In [84]:
# configure tissue section 1 =========
config_data_train['list_tissue']['anndata1']['file'] = train_file
#   the absolute path to anndata object of tissue section 1 on disk.

config_data_train['list_tissue']['anndata1']['obskey_cell_type'] = 'coarse_type'
#   meaning that for the 1st tissue section, cell type labels are provided in `broad_celltypes` column of `adata.obs`.

config_data_train['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = 'sid'
#   meaning that for the 1st tissue section, tissue section ID (i.e. slice ID) is provided in `sid` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_x'] = 'CenterX_global_px'
#   meaning that for the 1st tissue section, spatial x coordinates are provided in `CenterX_global_px` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_y'] = 'CenterY_global_px'
#   meaning that for the 1st tissue section, spatial y coordinates are provided in `CenterY_global_px` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_biological_batch_key'] = 'sid'
#   meaning that for the 1st tissue section, batch identifier is provided in `info_id` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['config_dataloader_train']['width_window'] = 1000
#   For tissue section one, the crop size of the customized dataloader desribed in Supplementary Fig. 16 of the paper.
#   The larger this number, the larger the tissue crops, and the bigger the subset of cells in each training iteration.
#      This implies that more GPU memory would be required during training.
#   In this notebook after calling `mintflow.setup_data` in Sec 4 the crop(s) are shown on tissue, 
#      with some information on image title which can help you tune this parameter.
#   In the manuscript we used `width_window` values between 300 and 800 depending on dataset.

config_data_train['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
    'n_neighs': 5,
    'set_diag': 'False',
    'delaunay': 'False',
}
#   The parameters for creating the neighbourhood graph for training tissue section 1

In [85]:
# configure tissue section 1 =======================
config_data_evaluation['list_tissue']['anndata1']['file'] = train_file
#   the absolute path to anndata object of tissue section 1 on disk.

config_data_evaluation['list_tissue']['anndata1']['obskey_cell_type'] = 'coarse_type'
#   meaning that for the 1st tissue section, cell type labels are provided in `broad_celltypes` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = 'sid'
#   meaning that for the 1st tissue section, tissue section ID (i.e. slice ID) is provided in `info_id` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_x'] = 'CenterX_global_px'
#   meaning that for the 1st tissue section, spatial x coordinates are provided in `x_centroid` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_y'] = 'CenterY_global_px'
#   meaning that for the 1st tissue section, spatial y coordinates are provided in `y_centroid` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_biological_batch_key'] = 'sid'
#   meaning that for the 1st tissue section, batch identifier is provided in `info_id` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['config_dataloader_test']['width_window'] = 1000
#   For tissue section one, the crop size of the customised dataloader desribed in Supplementary Fig. 16 of paper.
#   The larger this number, the larger the tissue crops, and the bigger the subset of cells in each training iteration.
#      This implies that more GPU memory would be required during training.
#   In this notebook after calling `mintflow.setup_data` in Sec 4 the crop(s) are shown on tissue, 
#      with some information on image title which can help you tune this parameter.
#   In the manuscript we used `width_window` values between 300 and 800 depending on dataset.

config_data_evaluation['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
    'n_neighs': 5,
    'set_diag': 'False',
    'delaunay': 'False',
}
#   The parameters for creating the neighbourhood graph for evaluation tissue section 1

In [86]:
config_data_train = mintflow.verify_and_postprocess_config_data_train(config_data_train) 

In [87]:
config_data_evaluation = mintflow.verify_and_postprocess_config_data_evaluation(config_data_evaluation)

In [88]:
config_model = mintflow.verify_and_postprocess_config_model(
    config_model,
    num_tissue_sections=1
)

 There is only one training tissue section --> the batch mixing coefficients `config_model['coef_xbarint2notbatchID_loss']` and `config_model['coef_xbarspl2notbatchID_loss']` were automatically set to 0.


In [89]:
config_training['num_training_epochs'] = 5
# number of training epochs, i.e. the number of times the model sees the dataset during training.

config_training['flag_enable_wandb'] = 'False'

In [90]:
config_training = mintflow.verify_and_postprocess_config_training(config_training)

In [ ]:
dict_all4_configs = {
    "config_data_train": config_data_train,
    "config_data_evaluation": config_data_evaluation,
    "config_model": config_model,
    "config_training": config_training,
}

data_mintflow = mintflow.setup_data(dict_all4_configs=dict_all4_configs)

In [ ]:
model = mintflow.setup_model(
    dict_all4_configs=dict_all4_configs,
    data_mintflow=data_mintflow
)

In [26]:
trainer = mintflow.Trainer(
    dict_all4_configs=dict_all4_configs,
    model=model,
    data_mintflow=data_mintflow
)

In [27]:
path_output_files = "/data2/a330d/datasets/mintflow_single_section/NonGit/Outputs_TutorialNotebook1"
os.makedirs(path_output_files, exist_ok=True)

In [28]:
for epoch in range(config_training["num_training_epochs"]):
    trainer.train_one_epoch()

    mintflow.dump_checkpoint(
        model=model,
        data_mintflow=data_mintflow,
        dict_all4_configs=dict_all4_configs,
        path_dump=os.path.join(path_output_files, "checkpoint_epoch_{}.pt".format(epoch)),
    )    

     Getting different embeddings to update the dual functions separately.


Tissue 0:   0%|          | 0/4774 [00:00<?, ?it/s]

Training the dual functions separately.:   0%|          | 0/200 [00:00<?, ?it/s]

MintFlow training epoch:   0%|          | 0/4202 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:869: UserWarning: The value argument must be within the support of the distribution
  ).log_prob(dict_qsamples['x_int'][:batch.batch_size])  # [b, num_genes]
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:893: UserWarning: The value argument must be within the support of the distribution
  ).log_prob(dict_qsamples['x_spl'][:batch.batch_size])  # [b, num_genes]


     Getting different embeddings to update the dual functions separately.


Tissue 0:   0%|          | 0/4774 [00:00<?, ?it/s]

Training the dual functions separately.:   0%|          | 0/200 [00:00<?, ?it/s]

MintFlow training epoch:   0%|          | 0/4202 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Eval

In [ ]:
# configure tissue section 1 =======================
config_data_evaluation['list_tissue']['anndata1']['file'] = train_file
#   the absolute path to anndata object of tissue section 1 on disk.

config_data_evaluation['list_tissue']['anndata1']['obskey_cell_type'] = 'coarse_type'
#   meaning that for the 1st tissue section, cell type labels are provided in `broad_celltypes` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = 'sid'
#   meaning that for the 1st tissue section, tissue section ID (i.e. slice ID) is provided in `info_id` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_x'] = 'CenterX_global_px'
#   meaning that for the 1st tissue section, spatial x coordinates are provided in `x_centroid` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_y'] = 'CenterY_global_px'
#   meaning that for the 1st tissue section, spatial y coordinates are provided in `y_centroid` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['obskey_biological_batch_key'] = 'sid'
#   meaning that for the 1st tissue section, batch identifier is provided in `info_id` column of `adata.obs`

config_data_evaluation['list_tissue']['anndata1']['config_dataloader_test']['width_window'] = 800
#   For tissue section one, the crop size of the customised dataloader desribed in Supplementary Fig. 16 of paper.
#   The larger this number, the larger the tissue crops, and the bigger the subset of cells in each training iteration.
#      This implies that more GPU memory would be required during training.
#   In this notebook after calling `mintflow.setup_data` in Sec 4 the crop(s) are shown on tissue, 
#      with some information on image title which can help you tune this parameter.
#   In the manuscript we used `width_window` values between 300 and 800 depending on dataset.

config_data_evaluation['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
    'n_neighs': 5,
    'set_diag': 'False',
    'delaunay': 'False',
}
#   The parameters for creating the neighbourhood graph for evaluation tissue section 1

In [ ]:
config_data_evaluation = mintflow.verify_and_postprocess_config_data_evaluation(
    config_data_evaluation
)

In [ ]:
predictions = mintflow.predict(
    device="cuda:0",
    dict_all4_configs=dict_all4_configs,
    data_mintflow=data_mintflow,
    model=model,
    evalulate_on_sections='all'
)

In [65]:
predictions.keys()

dict_keys(['TissueSection 0 (zero-based)'])

In [66]:
predictions = predictions['TissueSection 0 (zero-based)']

In [67]:
X_int = predictions['MintFlow_Xbar_int']
X_mic = predictions['MintFlow_Xbar_mic']

In [68]:
X_int.shape

(144002, 100)